# Naive RAG Pipeline — NBA Rules (Cohere)

This notebook implements a complete **Naive RAG** pipeline powered entirely by **Cohere**:

| Step | Cohere component |
|------|------------------|
| **Retrieval** | `embed-english-v3.0` — dense embeddings + cosine similarity |
| **Generation** | `command-r-plus` — grounded answer generation via Documents API |

### Pipeline Overview
```
JSON Chunks → Cohere Embeddings → Cosine Search → command-r-plus Answer → LLM as a judge
```

### Evaluation Metrics
We will utilise LLM as a judge and attach 3 criterias to it as follows
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

<h1>Preprocessing for Single Topic - NBA Rulebook</h1>

In [1]:
import json
import re

MIN_CHUNK_WORDS = 20

# ── regex ─────────────────────────────────────────────────────────────────────

RULE_RE     = re.compile(r'RULE\s+NO\.\s*(\d+[AB]?)[\s\u2014\-]+(.+)', re.I)
SECTION_RE  = re.compile(r'^Section\s+([IVXLC]+)[\s\u2014\-]+(.+)', re.I)
SUBSEC_RE   = re.compile(r'^([a-z])\.\s+(.+)', re.S)
COMMENTS_RE = re.compile(r'^COMMENTS ON THE RULES', re.I)

# ── helpers ───────────────────────────────────────────────────────────────────

def make_id(rule, section, sub=None):
    r = rule['number'].lower()
    s = section['number'].lower()
    return f"rule_{r}_sec_{s}_{sub}" if sub else f"rule_{r}_sec_{s}"

def flush_chunk(buf, rule, section, sub, page):
    if not buf or not rule or not section:
        return None

    text = ' '.join(buf).strip()
    text = re.sub(r' {2,}', ' ', text)

    # Drop chunks too short to be useful
    if not text or len(text.split()) < MIN_CHUNK_WORDS:
        return None

    # Prepend rule + section title so text is self-contained
    prefix = f"Rule {rule['number']} {rule['title']}, Section {section['title']}"
    if sub:
        prefix += f" ({sub})"

    return {
        'id':   make_id(rule, section, sub),
        'text': f"{prefix}: {text}",
    }


# ── main converter ────────────────────────────────────────────────────────────

def raw_to_naive_rag(raw: dict) -> dict:
    chunks = []

    current_rule    = None
    current_section = None
    current_sub     = None
    current_buf     = []
    current_page    = None

    for page_dict in raw['pages']:
        page_num = page_dict['page']

        for line in page_dict['text'].split('\n'):
            line = line.strip()
            if not line:
                continue

            # ── Rule header ───────────────────────────────────────────────
            m = RULE_RE.match(line)
            if m:
                chunk = flush_chunk(
                    current_buf, current_rule,
                    current_section, current_sub, current_page
                )
                if chunk:
                    chunks.append(chunk)
                current_rule    = {'number': m.group(1).upper(),
                                   'title':  m.group(2).strip()}
                current_section = None
                current_sub     = None
                current_buf     = []
                current_page    = page_num
                continue

            # ── Comments on the Rules ─────────────────────────────────────
            if COMMENTS_RE.match(line):
                chunk = flush_chunk(
                    current_buf, current_rule,
                    current_section, current_sub, current_page
                )
                if chunk:
                    chunks.append(chunk)
                current_rule    = {'number': 'COMMENTS',
                                   'title':  'Comments on the Rules'}
                current_section = None
                current_sub     = None
                current_buf     = []
                current_page    = page_num
                continue

            # ── Section header ────────────────────────────────────────────
            m = SECTION_RE.match(line)
            if m:
                chunk = flush_chunk(
                    current_buf, current_rule,
                    current_section, current_sub, current_page
                )
                if chunk:
                    chunks.append(chunk)
                current_section = {'number': m.group(1).upper(),
                                   'title':  m.group(2).strip()}
                current_sub     = None
                current_buf     = []
                current_page    = page_num
                continue

            # ── Lettered subsection ───────────────────────────────────────
            m = SUBSEC_RE.match(line)
            if m:
                chunk = flush_chunk(
                    current_buf, current_rule,
                    current_section, current_sub, current_page
                )
                if chunk:
                    chunks.append(chunk)
                current_sub  = m.group(1)
                current_page = page_num
                current_buf  = [m.group(2).strip()]
                continue

            # ── Everything else appends ───────────────────────────────────
            if current_section is not None:
                current_buf.append(line)

    # Final flush
    chunk = flush_chunk(
        current_buf, current_rule,
        current_section, current_sub, current_page
    )
    if chunk:
        chunks.append(chunk)

    # Deduplicate
    seen = {}
    for c in chunks:
        seen[c['id']] = c

    return {
        'total':  len(seen),
        'chunks': list(seen.values())
    }


# ── entry point ───────────────────────────────────────────────────────────────

if __name__ == '__main__':
    with open('nba_rules_raw.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    result = raw_to_naive_rag(raw)

    with open('nba_rules_naive_rag.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"Done — {result['total']} chunks written to nba_rules_naive_rag.json")

Done — 343 chunks written to nba_rules_naive_rag.json


## 0 · Install dependencies

In [2]:
# !pip install -q cohere numpy tqdm

## 1 · Imports & Configuration

In [14]:
import json, re, string, os, time
from collections import Counter

import numpy as np
import cohere
from tqdm import tqdm

# ── Cohere configuration ──────────────────────────────────────────────────────
EMBED_MODEL    = "embed-english-v3.0"   # embedding model for retrieval
LLM_MODEL      = "command-a-03-2025"       # LLM for answer generation
TOP_K          = 5                      # number of chunks to retrieve per query
JSON_PATH      = "nba_rules_naive_rag.json"

ACTIVE_DATASET = "test_questions_nba"

co = cohere.Client(api_key="bPxPN4MX7cZoTVNIK4zllf9tUGodwyYMTlKb64be")
print("✅ Cohere client ready")
print(f"   Embed model : {EMBED_MODEL}")
print(f"   LLM model   : {LLM_MODEL}")
print(f"   LLM model   : {JSON_PATH}")


✅ Cohere client ready
   Embed model : embed-english-v3.0
   LLM model   : command-a-03-2025
   LLM model   : nba_rules_naive_rag.json


## 2 · Load & Inspect the Knowledge Base

In [4]:
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

# Filter out table-of-contents stub chunks (text == "PAGE" or empty)
chunks = [c for c in raw["chunks"] if c["text"].strip() and c["text"].strip() != "PAGE"]

print(f"Total chunks in file : {raw['total']}")
print(f"Usable chunks        : {len(chunks)}")
print(f"\nSample chunk:")
print(json.dumps(chunks[10], indent=2))

Total chunks in file : 343
Usable chunks        : 343

Sample chunk:
{
  "id": "rule_1_sec_ii_c",
  "text": "Rule 1 COURT DIMENSIONS\u2014EQUIPMENT, Section Equipment (c): Home management is required to have a spare board with supporting unit on hand for emergencies, and a steel tape or extension ruler and a level for use if necessary."
}


## 3 · Build the Dense Retriever with Cohere Embeddings

We embed every chunk once upfront and store the vectors in memory.
At query time we embed the question and compute cosine similarity against the corpus.


In [5]:
def embed_texts(texts: list[str], input_type: str, batch_size: int = 96) -> np.ndarray:
    """
    Embed a list of texts in batches using Cohere embed-english-v3.0.

    input_type:
        'search_document' — for the knowledge-base corpus
        'search_query'    — for the user query at retrieval time
    """
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding ({input_type})"):
        batch = texts[i : i + batch_size]
        response = co.embed(
            texts=batch,
            model=EMBED_MODEL,
            input_type=input_type,
        )
        all_embeddings.extend(response.embeddings)
    return np.array(all_embeddings, dtype=np.float32)


# Use text so embeddings capture document structure
corpus_texts = [f"{c['text']}" for c in chunks]

# Embed the full corpus once (this may take ~1-2 min for ~400 chunks)
corpus_embeddings = embed_texts(corpus_texts, input_type="search_document")

print(f"\nCorpus embedding matrix : {corpus_embeddings.shape}")
print(f"Embedding dimension     : {corpus_embeddings.shape[1]}")


Embedding (search_document): 100%|██████████| 4/4 [00:04<00:00,  1.09s/it]


Corpus embedding matrix : (343, 1024)
Embedding dimension     : 1024


In [6]:
def cosine_scores(query_vec: np.ndarray, corpus_vecs: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between one query vector and the corpus matrix."""
    q_norm = query_vec  / (np.linalg.norm(query_vec) + 1e-10)
    c_norm = corpus_vecs / (np.linalg.norm(corpus_vecs, axis=1, keepdims=True) + 1e-10)
    return c_norm @ q_norm


def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """Embed query and return the top-k most relevant chunks."""
    resp      = co.embed(texts=[query], model=EMBED_MODEL, input_type="search_query")
    query_vec = np.array(resp.embeddings[0], dtype=np.float32)

    scores  = cosine_scores(query_vec, corpus_embeddings)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        chunk = chunks[idx].copy()
        chunk["score"] = float(scores[idx])
        results.append(chunk)
    return results


# Sanity check
sample = retrieve("how many fouls before a player is ejected")
for r in sample:
    print(f"[{r['score']:.4f}] {r}")

[0.6038] {'id': 'rule_3_sec_i_b', 'text': 'Rule 3 PLAYERS, SUBSTITUTES, AND COACHES, Section Team (b): In the event that there are only five eligible players remaining and one of these players is injured and must leave the game or is ejected, he must be replaced by the last player who was disqualified by reason of receiving six personal fouls. Each subsequent requirement to replace an injured or ejected player will be treated in this inverse order. Any such re-entry into a game by a disqualified player shall be penalized by a technical foul.', 'score': 0.6037944555282593}
[0.6003] {'id': 'rule_3_sec_i_a', 'text': 'Rule 3 PLAYERS, SUBSTITUTES, AND COACHES, Section Team (a): Each team shall consist of five players. A player is disqualified from the game when he receives his sixth personal foul. No team may be reduced to less than five players. If a player in the game receives his sixth personal foul and all substitutes have already been disqualified, said player shall remain in the game 

## 4 · LLM Answer Generation with `command-r-plus`

We use Cohere's **Chat** endpoint with the `documents` parameter.
This passes retrieved chunks as structured grounding context, which reduces hallucination
and keeps answers faithful to the rulebook.

In [7]:
PREAMBLE = """\
You are an expert NBA rules assistant.
Answer the user's question using ONLY the provided document excerpts from the NBA rulebook.
Be concise and factual. If the answer is not in the documents, say "Not found in the provided context."
"""


def generate_answer(query: str, retrieved_chunks: list[dict]) -> str:
    """Call command-r-plus with retrieved chunks passed as grounded documents."""
    # Cohere's Chat API accepts a 'documents' list for grounded RAG generation
    documents = [
        {
            "id"      : c["id"],
            "title"   : c["id"],
            "snippet" : c["text"],
        }
        for c in retrieved_chunks
    ]

    response = co.chat(
        model=LLM_MODEL,
        preamble=PREAMBLE,
        message=query,
        documents=documents,
        temperature=0,
        max_tokens=300,
    )
    return response.text.strip()


def rag_answer(query: str, top_k: int = TOP_K) -> tuple[str, list[dict]]:
    """Full RAG pipeline: retrieve → generate."""
    retrieved = retrieve(query, top_k=top_k)
    answer    = generate_answer(query, retrieved)
    return answer, retrieved


# Demo
q = "How many timeouts does each team get per half?"
ans, ctx = rag_answer(q)
print(f"Q: {q}\nA: {ans}")

Q: How many timeouts does each team get per half?
A: Each team is entitled to seven (7) charged timeouts during regulation play.


## 5 · Define Evaluation Dataset

Each entry has a **question** and a short **reference answer** (ground truth).

> 💡 Add more QA pairs here to make the evaluation more robust.

In [9]:
with open(f"{ACTIVE_DATASET}.json", "r") as f:
    eval_dataset = json.load(f)
    
print(f"Evaluation dataset size: {len(eval_dataset)} questions")



Evaluation dataset size: 30 questions


## 6 · LLM as a judge

In [10]:
test_questions = eval_dataset

In [11]:
def chat_with_retry(max_retries=4, **kwargs):
    """Wraps co.chat with exponential backoff on TooManyRequestsError."""
    for attempt in range(max_retries):
        try:
            return co.chat(**kwargs)
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 15 * (2 ** attempt)  # 15s, 30s, 60s, 120s
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise  # re-raise non-rate-limit errors immediately
    raise RuntimeError("Max retries exceeded on Cohere rate limit.")

In [12]:
# ── Step 2: LLM Judge ──────────────────────────────────────────────────────
# Scores a single (question, context, answer) triple on 3 metrics.

def llm_judge(question, context, answer_text):
    """Score an answer using an LLM judge.

    Returns a dict with scores (1–5) for:
        faithfulness      – every claim is supported by the context
        answer_relevance  – answer directly addresses the question
        context_relevance – retrieved context was useful for the question

    A reasoning string is also returned for transparency.
    """
    prompt = f"""You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems.

Score the answer below on three criteria using a 1–5 integer scale:

  1 = very poor  |  3 = acceptable  |  5 = excellent

Criteria:
- faithfulness:      Every claim in the answer is directly supported by the provided context.
                     Penalise heavily for hallucination or any statement not grounded in the context.
- answer_relevance:  The answer directly and completely addresses the question with a real answer.
                     IMPORTANT: If the answer says it "cannot answer", "no information", "not found in
                     the graph", or anything similar — that is a FAILED answer. Score it 1.
- context_relevance: The retrieved context actually contained information useful for answering
                     the question. If the context is off-topic or the answer says no info was found,
                     score this 1.

QUESTION:
{question}

RETRIEVED CONTEXT:
{context if context else "(no context retrieved)"}

ANSWER:
{answer_text}

Return ONLY a valid JSON object (no extra text):
{{
  "faithfulness": <1-5>,
  "answer_relevance": <1-5>,
  "context_relevance": <1-5>,
  "reasoning": "<one sentence explaining the scores>"
}}
"""
    response = chat_with_retry(model="command-r7b-12-2024", message=prompt, temperature=0)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1].lstrip("json").strip()
    try:
        scores = json.loads(raw)
        # Clamp scores to valid range
        for key in ("faithfulness", "answer_relevance", "context_relevance"):
            scores[key] = max(1, min(5, int(scores.get(key, 1))))
        return scores
    except (json.JSONDecodeError, ValueError):
        return {"faithfulness": 0, "answer_relevance": 0, "context_relevance": 0,
                "reasoning": "parse error"}

# Quick smoke test
_test = llm_judge(
    "What is machine learning?",
    "Machine learning -[is part of]-> Artificial intelligence",
    "Machine learning is a subfield of artificial intelligence."
)
print("Judge smoke test:", _test)

Judge smoke test: {'faithfulness': 5, 'answer_relevance': 5, 'context_relevance': 5, 'reasoning': 'The answer is directly supported by the context, is relevant to the question, and provides a complete and accurate response.'}


In [15]:
# ── Step 3: Run evaluation ─────────────────────────────────────────────────
# Saves results after every question so a crash won't lose progress.
# Re-running this cell will skip already-completed questions automatically.

EVAL_RESULTS_PATH = f"data/eval_results_{ACTIVE_DATASET}.json"

# Resume from existing results if available
if os.path.exists(EVAL_RESULTS_PATH):
    with open(EVAL_RESULTS_PATH, "r", encoding="utf-8") as f:
        eval_results = json.load(f)
    done_questions = {r["question"] for r in eval_results}
    print(f"Resuming — {len(eval_results)} already done, {len(test_questions) - len(done_questions)} remaining.")
else:
    eval_results = []
    done_questions = set()

for i, item in enumerate(test_questions):
    question = item["question"]

    if question in done_questions:
        print(f"[{i+1}/{len(test_questions)}] Skipping (already done): {question[:60]}")
        continue

    print(f"[{i+1}/{len(test_questions)}] {question}")

    for attempt in range(3):
        try:
            result    = rag_answer(question)
            ans       = result[0]
            retrieved = result[1]                          # raw chunks
            ctx       = " | ".join(c["text"] for c in retrieved)  # string for llm_judge
            scores    = llm_judge(question, ctx, ans)
            break
        except Exception as e:
            if "TooManyRequests" in type(e).__name__ or "429" in str(e):
                wait = 20 * (attempt + 1)
                print(f"  Rate limited — waiting {wait}s (attempt {attempt + 1}/3)...")
                time.sleep(wait)
            else:
                raise
    else:
        print(f"  Skipping after 3 failed attempts.")
        continue

    reference = item.get("reference_answer") or item.get("reference", "")

    eval_results.append({
        "question":          question,
        "reference_answer":  reference,
        "naiveRAG_answer":   ans,
        "context":           retrieved,
        "faithfulness":      scores["faithfulness"],
        "answer_relevance":  scores["answer_relevance"],
        "context_relevance": scores["context_relevance"],
        "reasoning":         scores.get("reasoning", ""),
    })

    with open(EVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(eval_results, f, indent=2, ensure_ascii=False)

    time.sleep(4)

print(f"\nDone. {len(eval_results)}/{len(test_questions)} results saved to {EVAL_RESULTS_PATH}")

# ── Summary table ──────────────────────────────────────────────────────────
metrics = ("faithfulness", "answer_relevance", "context_relevance")
averages = {m: sum(r[m] for r in eval_results) / len(eval_results) for m in metrics}

print(f"\n{'='*45}")
print(f"  LLM as a judge Evaluation  —  dataset: {ACTIVE_DATASET}")
print(f"{'='*45}")
print(f"  Questions evaluated : {len(eval_results)}")
print(f"  Faithfulness        : {averages['faithfulness']:.2f} / 5")
print(f"  Answer Relevance    : {averages['answer_relevance']:.2f} / 5")
print(f"  Context Relevance   : {averages['context_relevance']:.2f} / 5")
avg_overall = sum(averages.values()) / len(averages)
print(f"  Overall             : {avg_overall:.2f} / 5")
print(f"{'='*45}")

# Per-question breakdown
print("\nPer-question scores (F=Faithfulness, A=Answer Rel., C=Context Rel.")
print(f"{'#':<4} {'F':>3} {'A':>3} {'C':>3} Question")
print("-" * 70)
for i, r in enumerate(eval_results, 1):
    q_preview = r["question"][:48] + "…" if len(r["question"]) > 48 else r["question"]
    print(f"{i:<4} {r['faithfulness']:>3} {r['answer_relevance']:>3} {r['context_relevance']:>3} {q_preview}")

Resuming — 1 already done, 29 remaining.
[1/30] Skipping (already done): How many personal fouls does it take to foul out of an NBA g
[2/30] How long is the shot clock in the NBA?
[3/30] How many timeouts is each team allowed per game?
[4/30] How many players are on each team on the court at one time?
[5/30] What is the duration of each quarter in an NBA game?
[6/30] How far is the free throw line from the backboard?
[7/30] How many seconds does a team have to advance the ball past half court?
[8/30] What is the penalty for a technical foul in terms of free throws?
[9/30] How many challenges does each coach get per game?
[10/30] What happens when a player commits a flagrant foul penalty 2?
[11/30] Skipping (already done): How many personal fouls does it take to foul out of an NBA g
[12/30] How long is the shot clock in the NBA?
[13/30] How many timeouts is each team allowed per game?
[14/30] How many players are on each team on the court at one time?
[15/30] What is the duration of eac